In [88]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import re

import pdfplumber

import os
import camelot
from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

from docx import Document  # Import the Document class from the docx module to work with Word documents
import pandas as pd 
import pypandoc

In [89]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'QA QCB' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running QA QCB Web Scraping Tool v.1.0


In [90]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [91]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName+' 1': 'https://www.qcb.gov.qa/EN/Pages/Publication.aspx?IndexSelect=1',

        }



Typology={

        regulatorName+' 1': 'Banks',
        regulatorName+' 2': 'Investment, Finance & Financial Consulting Companies',
        regulatorName+' 3': 'Licensed Exchange Houses',
        regulatorName+' 4': 'Investment Funds',
        regulatorName+' 5': 'Insurance Companies',
        regulatorName+' 6': 'Auxiliary Insurance Service Providers',
        regulatorName+' 7': 'Payment Service Providers',


        }




sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}




now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [92]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [ ]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    
    
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(2)

    supervision_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Supervision')]")
    driver.execute_script("arguments[0].click();", supervision_button)
    sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(2)
    panel_ = soup.find(id='v-pills-home0')
    links = panel_.find_all('ul')

    
    for link in links:
        regulatorCode = None

        if 'Directory of Auxiliary' in link.text:
                regulatorCode = '6'
                href = link.find('a')['href']
                download_button = driver.find_element(By.XPATH, f"//a[@href='{href}']/div")
                driver.execute_script("arguments[0].click();", download_button)
                print('Deal with ' + regulatorCode)
                sleep(3)
                pdf_file = os.listdir(tempfolder)[0]
                filePath = os.path.join(tempfolder, pdf_file)
                tables = []
                with pdfplumber.open(filePath) as pdf:
                    for page in pdf.pages:
                        table = page.extract_tables({})
                        tables.append(table)
                cleaned_tables = []
                for table in tables:
                    cleaned_table = []
                    for row in table:
                        for r in row:
                            cleaned_row = [cell for cell in r if cell is not None]
                            if cleaned_row:  # Only add non-empty rows
                                cleaned_table.append(cleaned_row)
                    
                    cleaned_tables.append(cleaned_table)

                for clean_table in cleaned_tables:
                    for table in range(len(clean_table)):
                        if clean_table[table][0].isdigit():
                            #print(clean_table[table])
                            cleaned_list = [ele.replace('\n',' ') for ele in clean_table[table]]
                            cleaned_list = [ele.replace('-','') for ele in clean_table[table]]
                            name = cleaned_list[1]
                            address = cleaned_list[2]
                            tel_ = cleaned_list[3]
                            fax_ = cleaned_list[4]
                            web_ = cleaned_list[6]
                            email_ = cleaned_list[7]
                            sqldict['Name'].append(name)
                            sqldict['Address_1'].append(address)
                            sqldict['Phone'].append(tel_)
                            sqldict['Fax'].append(fax_)
                            sqldict['Website'].append(web_)
                            sqldict['Email'].append(email_)
                            sqldict['RegulationType'].append('Regulated')
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['ListCode'].append(regulatorCode)
                            sqldict['RegCode'].append('QCB')
                            sqldict['RegCtry'].append('QA')
                            sqldict['ListName'].append(Typology[regulatorName + ' '+regulatorCode])
                            sqldict = bourange_same_length_array(sqldict)
                os.remove(filePath)
            
            
            
        elif 'Directory of Banks' in link.text:
            regulatorCode = '1'
            href = link.find('a')['href']
            download_button = driver.find_element(By.XPATH, f"//a[@href='{href}']/div")
            driver.execute_script("arguments[0].click();", download_button)
            print('Deal with ' + regulatorCode)
            sleep(3)
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
            table_settings = {
            "vertical_strategy": "lines",  # Use explicitly defined vertical lines
            "horizontal_strategy": "explicit",    # X-coordinates of vertical lines
            "explicit_horizontal_lines": 
            [25,40,60,80,100,120,140,170,190,210,230,250,270,280,295,310,320,340,360,380,400,420,440,460,480,505]}

            tables = []
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table(table_settings=table_settings)
                    tables.append(table)
            for table in tables:
                for tab in table:

                    if tab[0].isdigit():
                        
                        cleaned_list = [ele.replace('\n',' ') for ele in tab]
                        cleaned_list = [ele.replace('-',' ') for ele in tab]
                        name = cleaned_list[1]
                        address_ = cleaned_list[2]
                        tel_ = cleaned_list[3]
                        fax_ = cleaned_list[4]
                        web_ = cleaned_list[6]
                        email_ = cleaned_list[7]


                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(address_.replace('\n',' '))
                        sqldict['Phone'].append(tel_)
                        sqldict['Fax'].append(fax_)
                        sqldict['Website'].append(web_)
                        sqldict['Email'].append(email_)
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append(regulatorCode)
                        sqldict['RegCode'].append('QCB')
                        sqldict['RegCtry'].append('QA')
                        sqldict['ListName'].append(Typology[regulatorName + ' '+regulatorCode])
                        sqldict = bourange_same_length_array(sqldict)
            try:
                os.remove(filePath)
            except:
                pass
            
        elif 'Investment and finance companies' in link.text:
            regulatorCode = '2'
            href = link.find('a')['href']
            download_button = driver.find_element(By.XPATH, f"//a[@href='{href}']/div")
            driver.execute_script("arguments[0].click();", download_button)
            sleep(3)
            print('Deal with ' + regulatorCode)
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
            document = Document(filePath)

            # Initialize an empty list to store tables
            tables = []

            # Iterate through each table in the document
            for table in document.tables:
                # Create a DataFrame structure with empty strings, sized by the number of rows and columns in the table
                df = [['' for _ in range(len(table.columns))] for _ in range(len(table.rows))]
                
                # Iterate through each row in the current table
                for i, row in enumerate(table.rows):
                    # Iterate through each cell in the current row
                    for j, cell in enumerate(row.cells):
                        # If the cell has text, store it in the corresponding DataFrame position
                        if cell.text:
                            df[i][j] = cell.text
                
                # Convert the list of lists (df) to a pandas DataFrame and add it to the tables list
                tables.append(pd.DataFrame(df))
                tables = pd.concat(tables, axis=0, ignore_index=True)
                tables.columns =  tables.iloc[0]
                tables = tables[1:]
                for tab in tables.itertuples():
                    if str(tab[1]).isdigit():
                        name = tab[2]
                        license_number = tab[3]
                        license_date = tab[4]
                        address_ = tab[5]

                        tel_ = tab[6]
                        fax_ = tab[7]
                        web_ = tab[9]
                        email_ = tab[10]
                        #print(name, license_number, license_date, address_, tel_, fax_, web_, email_)
                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(address_)
                        sqldict['Phone'].append(tel_)
                        sqldict['Fax'].append(fax_)
                        sqldict['Website'].append(web_)
                        sqldict['Email'].append(email_)
                        sqldict['InternalID_1'].append(license_number)
                        sqldict['InternalID_1_type'].append('License Number')
                        sqldict['RegulationDate'].append(license_date)
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append(regulatorCode)
                        sqldict['RegCode'].append('QCB')
                        sqldict['RegCtry'].append('QA')
                        sqldict['ListName'].append(Typology[regulatorName + ' '+regulatorCode])
                        sqldict = bourange_same_length_array(sqldict)
                os.remove(filePath)

                
        elif 'Exchange House' in link.text:
                regulatorCode = '3'
                href = link.find('a')['href']
                download_button = driver.find_element(By.XPATH, f"//a[@href='{href}']/div")
                driver.execute_script("arguments[0].click();", download_button)
                sleep(3)
                print('Deal with ' + regulatorCode)
                pdf_file = os.listdir(tempfolder)[0]
                filePath = os.path.join(tempfolder, pdf_file)
                document = Document(filePath)

                # Initialize an empty list to store tables
                tables = []

                # Iterate through each table in the document
                for table in document.tables:
                    # Create a DataFrame structure with empty strings, sized by the number of rows and columns in the table
                    df = [['' for _ in range(len(table.columns))] for _ in range(len(table.rows))]
                    
                    # Iterate through each row in the current table
                    for i, row in enumerate(table.rows):
                        # Iterate through each cell in the current row
                        for j, cell in enumerate(row.cells):
                            # If the cell has text, store it in the corresponding DataFrame position
                            if cell.text:
                                df[i][j] = cell.text
                    
                    # Convert the list of lists (df) to a pandas DataFrame and add it to the tables list
                    tables.append(pd.DataFrame(df))
                    tables = pd.concat(tables, axis=0, ignore_index=True)
                    tables.columns =  tables.iloc[0]
                    tables = tables[1:]
                    sqldict['Name'].extend(tables.iloc[:,1].tolist())
                    sqldict['Address_1'].extend(tables.iloc[:,2].tolist())
                    sqldict['Phone'].extend(tables.iloc[:,3].tolist())
                    sqldict['Fax'].extend(tables.iloc[:,4].tolist())
                    sqldict['Website'].extend(tables.iloc[:,6].tolist())
                    sqldict['Email'].extend(tables.iloc[:,7].tolist())


                    # Use extend with repeated values for RegulationType
                    sqldict['RegulationType'].extend(['Regulated'] * len(tables.iloc[:, 1].tolist()))
                    sqldict['ListProcessDate'].extend([processdate] * len(tables.iloc[:, 1].tolist()))
                    sqldict['ListCode'].extend([regulatorCode] * len(tables.iloc[:, 1].tolist()))
                    sqldict['RegCode'].extend(['QCB'] * len(tables.iloc[:, 1].tolist()))
                    sqldict['RegCtry'].extend(['QA'] * len(tables.iloc[:, 1].tolist()))
                    sqldict['ListName'].extend([Typology[regulatorName + ' ' + regulatorCode]] * len(tables.iloc[:, 1].tolist()))
                    sqldict = bourange_same_length_array(sqldict)
                    os.remove(filePath)

        elif 'Investment Funds' in link.text:
                regulatorCode = '4'
                href = link.find('a')['href']
                download_button = driver.find_element(By.XPATH, f"//a[@href='{href}']/div")
                driver.execute_script("arguments[0].click();", download_button)
                sleep(3)
                print('Deal with ' + regulatorCode)
                pdf_file = os.listdir(tempfolder)[0]
                filePath = os.path.join(tempfolder, pdf_file)
                table = camelot.read_pdf(filePath)
                df_table = table[0].df
                df_table = df_table[1:]

                for i, j in zip(df_table.iloc[:, 1], df_table.iloc[:, 3]):

                    sqldict['Name'].append(i.replace('\n', ' '))
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationDate'].append(j.replace('\n', ' '))

                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListCode'].append(regulatorCode)
                    sqldict['RegCode'].append('QCB')
                    sqldict['RegCtry'].append('QA')
                    sqldict['ListName'].append(Typology[regulatorName + ' '+regulatorCode])
                    sqldict = bourange_same_length_array(sqldict)
                    try:
                        os.remove(filePath)
                    except:
                        pass

        elif 'Insurance Companies' in link.text:
                regulatorCode = '5'
                href = link.find('a')['href']
                download_button = driver.find_element(By.XPATH, f"//a[@href='{href}']/div")
                driver.execute_script("arguments[0].click();", download_button)
                sleep(3)
                print('Deal with ' + regulatorCode)
                pdf_file = os.listdir(tempfolder)[0]
                filePath = os.path.join(tempfolder, pdf_file)
                table = camelot.read_pdf(filePath)
                df_table = table[0].df
                df_table = df_table[1:]
                df_table = df_table.iloc[:,1:]


                for name, address_,tel_,fax_,web_,email_ in zip(df_table.iloc[:, 0],df_table.iloc[:, 1],df_table.iloc[:, 2], df_table.iloc[:, 3],df_table.iloc[:, 5],df_table.iloc[:, 6]):

                                    sqldict['Name'].append(name)
                                    sqldict['Address_1'].append(address_)
                                    sqldict['Phone'].append(tel_)
                                    sqldict['Fax'].append(fax_)
                                    sqldict['Website'].append(web_)
                                    sqldict['Email'].append(email_)
                                    sqldict['ListProcessDate'].append(processdate)

                                    sqldict['RegulationType'].append('Regulated')
                                    sqldict['ListCode'].append(regulatorCode)
                                    sqldict['RegCode'].append('QCB')
                                    sqldict['RegCtry'].append('QA')
                                    sqldict['ListName'].append(Typology[regulatorName + ' '+regulatorCode])
        elif 'Payment Service' in link.text:
                regulatorCode = '7'
                href = link.find('a')['href']
                download_button = driver.find_element(By.XPATH, f"//a[@href='{href}']/div")
                driver.execute_script("arguments[0].click();", download_button)
                sleep(3)
                print('Deal with ' + regulatorCode)
                pypandoc.download_pandoc()

                pdf_file = os.listdir(tempfolder)[0]
                filePath = os.path.join(tempfolder, pdf_file)

                # Path to save the DOCX file
                docx_file = os.path.join(tempfolder, "output.docx")

                # Convert RTF to DOCX
                pypandoc.convert_file(filePath, 'docx', outputfile=docx_file)

                print(f"RTF file has been successfully converted to DOCX and saved as {docx_file}.")

                document = Document(docx_file)
                tables = []

                # Iterate through each table in the document
                for table in document.tables:
                    # Create a DataFrame structure with empty strings, sized by the number of rows and columns in the table
                    df = [['' for _ in range(len(table.columns))] for _ in range(len(table.rows))]
                    
                    # Iterate through each row in the current table
                    for i, row in enumerate(table.rows):
                        # Iterate through each cell in the current row
                        for j, cell in enumerate(row.cells):
                            # If the cell has text, store it in the corresponding DataFrame position
                            if cell.text:
                                df[i][j] = cell.text
                    
                    # Convert the list of lists (df) to a pandas DataFrame and add it to the tables list
                    tables.append(pd.DataFrame(df))
                    tables = pd.concat(tables, axis=0, ignore_index=True)
                    tables.columns =  tables.iloc[0]
                    tables = tables[2:]
                    tables = tables[tables.iloc[:,1]!='']

                    sqldict['Name'].extend(tables.iloc[:,1].tolist())
                    sqldict['Address_1'].extend(tables.iloc[:,2].tolist())
                    sqldict['Phone'].extend(tables.iloc[:,3].tolist())
                    sqldict['Email'].extend(tables.iloc[:,4].tolist())

                    # Use extend with repeated values for RegulationType
                    sqldict['RegulationType'].extend(['Regulated'] * len(tables.iloc[:, 1].tolist()))
                    sqldict['ListProcessDate'].extend([processdate] * len(tables.iloc[:, 1].tolist()))
                    sqldict['ListCode'].extend([regulatorCode] * len(tables.iloc[:, 1].tolist()))
                    sqldict['RegCode'].extend(['QCB'] * len(tables.iloc[:, 1].tolist()))
                    sqldict['RegCtry'].extend(['QA'] * len(tables.iloc[:, 1].tolist()))
                    sqldict['ListName'].extend([Typology[regulatorName + ' ' + regulatorCode]] * len(tables.iloc[:, 1].tolist()))
                    sqldict = bourange_same_length_array(sqldict)
                    os.remove(filePath)

        

driver.quit()

    


Working with list QA QCB 1
Deal with 7
RTF file has been successfully converted to DOCX and saved as C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\QA QCB\tempfolder\output.docx.
Deal with 2
Deal with 3
Deal with 1
Deal with 4
Deal with 5
Deal with 6


In [85]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df[df['Name']!='']

df['Address_1'] = df['Address_1'].str.replace('\n', ' ')
df['Name'] = df['Name'].str.replace('\n', ' ')


df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
     

C:\Users\wuj1\AppData\Local\Temp\ipykernel_27424\3483721940.py:16: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
import subprocess
import sys

required_packages = ['python-docx',  'pypandoc']

for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f'Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

In [86]:
df.to_csv('qa_qcb_total_v2.csv')

In [80]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 118 values.
Key 'priority' has 118 values.
Key 'ListLabel' has 118 values.
Key 'Typology' has 118 values.
Key 'EntryType' has 118 values.
Key 'Name' has 118 values.
Key 'InternalID_1' has 118 values.
Key 'InternalID_1_type' has 118 values.
Key 'InternalID_2' has 118 values.
Key 'InternalID_2_type' has 118 values.
Key 'InternalID_3' has 118 values.
Key 'InternalID_3_type' has 118 values.
Key 'CoType' has 118 values.
Key 'License_Type' has 118 values.
Key 'Address_1' has 118 values.
Key 'Address_2' has 118 values.
Key 'City' has 118 values.
Key 'Zip' has 118 values.
Key 'Cntry' has 118 values.
Key 'Phone' has 118 values.
Key 'Fax' has 118 values.
Key 'Website' has 118 values.
Key 'Email' has 118 values.
Key 'RegulationType' has 118 values.
Key 'RegulationTypeCode' has 118 values.
Key 'RegulationDate' has 118 values.
Key 'CancellationDate' has 118 values.
Key 'RegCtry' has 118 values.
Key 'RegCode' has 118 values.
Key 'ListCode' has 118 values.
Key 'ListLanguage' has 118 v

In [36]:
import pypandoc

# Ensure pandoc is available
pypandoc.download_pandoc()

pdf_file = os.listdir(tempfolder)[0]
filePath = os.path.join(tempfolder, pdf_file)

# Path to save the DOCX file
docx_file = os.path.join(tempfolder, "output.docx")

# Convert RTF to DOCX
pypandoc.convert_file(filePath, 'docx', outputfile=docx_file)

print(f"RTF file has been successfully converted to DOCX and saved as {docx_file}.")

document = Document(docx_file)

RTF file has been successfully converted to DOCX and saved as C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\QA QCB\tempfolder\output.docx.


In [ ]:
tables = []

# Iterate through each table in the document
for table in document.tables:
    # Create a DataFrame structure with empty strings, sized by the number of rows and columns in the table
    df = [['' for _ in range(len(table.columns))] for _ in range(len(table.rows))]
    
    # Iterate through each row in the current table
    for i, row in enumerate(table.rows):
        # Iterate through each cell in the current row
        for j, cell in enumerate(row.cells):
            # If the cell has text, store it in the corresponding DataFrame position
            if cell.text:
                df[i][j] = cell.text
    
    # Convert the list of lists (df) to a pandas DataFrame and add it to the tables list
    tables.append(pd.DataFrame(df))
    tables = pd.concat(tables, axis=0, ignore_index=True)
    tables.columns =  tables.iloc[0]
    tables = tables[2:]
    tables = tables[tables.iloc[:,1]!='']

    sqldict['Name'].extend(tables.iloc[:,1].tolist())
    sqldict['Address_1'].extend(tables.iloc[:,2].tolist())
    sqldict['Phone'].extend(tables.iloc[:,3].tolist())
    sqldict['Email'].extend(tables.iloc[:,4].tolist())

    # Use extend with repeated values for RegulationType
    sqldict['RegulationType'].extend(['Regulated'] * len(tables.iloc[:, 1].tolist()))
    sqldict['ListProcessDate'].extend([processdate] * len(tables.iloc[:, 1].tolist()))
    sqldict['ListCode'].extend([regulatorCode] * len(tables.iloc[:, 1].tolist()))
    sqldict['RegCode'].extend(['QCB'] * len(tables.iloc[:, 1].tolist()))
    sqldict['RegCtry'].extend(['QA'] * len(tables.iloc[:, 1].tolist()))
    sqldict['ListName'].extend([Typology[regulatorName + ' ' + regulatorCode]] * len(tables.iloc[:, 1].tolist()))
    sqldict = bourange_same_length_array(sqldict)
    os.remove(filePath)